In [30]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from helper_functions.feature_transformer import *
from helper_functions.visualizations import *
from helper_functions.imputer import CustomIterativeImputer

data = pd.read_csv("../../preliminar_phase_submision/training_data/SyntheticData_Training.csv")
dictionary = pd.read_csv("tmp_datasets/DataDictionary.csv")
target = "inhospital_mortality"

multi_imputer = CustomIterativeImputer()
X = data.drop(columns=['inhospital_mortality', 'studyid_adm', 'lengthadm'],
               errors='ignore')
y = data["inhospital_mortality"]

In [ ]:
primary_features_keys = {
    # for calculating bmi z-scores
    'agecalc_adm',
    'height_cm_adm',
    'weight_kg_adm',
    'sex_adm',

    # caprefill is already a binary value
    'caprefill_adm',

    # 1 if glucose < 4 mmol/L (hypoglycemia) or > 10 mmol/L (hyperglycemia)
    # 0 if glucose is normal (4–10 mmol/L).
    'glucose_mmolpl_adm',

    # 1 if hematocrit < 30% (anemia).
    # 0 if hematocrit ≥ 30%
    'hematocrit_gpdl_adm',

    # Normal Lactate: <2.0 mmol/L, Elevated Lactate: ≥2.0 mmol/L (suggests tissue hypoxia or impaired clearance)
    # but the clearance and delta are more indicative than single results
    'lactate_mmolpl_adm',
    'lactate2_mmolpl_adm',

    # Normal SpO2: ≥95% on room air. Concerning SpO2: <90% (hypoxemia).
    'spo2site1_pc_oxi_adm',
    'spo2site2_pc_oxi_adm',

    # Use spo2site1_pc_oxi_adm or spo2site2_pc_oxi_adm with 
    # the highest sqi1_perc_oxi_adm or sqi2_perc_oxi_adm for reliable SpO2.
    # If the SQIs are equal, Use the lower SpO2 reading (to be more conservative)
    'sqi1_perc_oxi_adm',
    'sqi2_perc_oxi_adm',

    # see secondary features
    'sysbp_mmhg_adm',
    'diasbp_mmhg_adm',
    'hr_bpm_adm',

    # 1 if temperature > 38°C (fever) or < 36°C (hypothermia).
    # 0 if temperature is normal (36–38°C)
    'temp_c_adm'}

row = X[list(primary_features_keys)].dropna().sample(1)
row[target] = y.loc[X.index]

primary_features = {}
for key in primary_features_keys:
    primary_features[key] = row[key].values[0]

primary_features[target] = row[target]



secondary_features = {
    'bmi_z_score' : primary_features['weight_kg_adm'] / primary_features['height_cm_adm']**2, # cross-referenced to chart
    # A higher shock index indicates worse perfusion and is associated with shock.
    'shock_index' : (primary_features['hr_bpm_adm'] / primary_features['sysbp_mmhg_adm']),
    # MAP < 65 mmHg is a sign of hypotension and poor perfusion.
    'map' : primary_features['diasbp_mmhg_adm'] + primary_features['sysbp_mmhg_adm']/3 - primary_features['diasbp_mmhg_adm'],
    # lactate clearance < 10%: Associated with poor outcomes, including higher mortality and organ failure. 
    # a negative value indicates rising lactate (worsening tissue hypoxia) 
    'lactate_clearance' : 100 * (primary_features['lactate_mmolpl_adm'] - primary_features['lactate2_mmolpl_adm']) / primary_features['lactate_mmolpl_adm'],
    # a drop in SpO2 (e.g., from 96% to 90%) over a short period is alarming.
    'spo_delta': (primary_features['spo2site1_pc_oxi_adm'] - primary_features['spo2site2_pc_oxi_adm'])
}

for key, value in secondary_features.items():
    row[key] = value

display(row.T)

,368
temp_c_adm,38.0
hematocrit_gpdl_adm,42.0
weight_kg_adm,14.0
hr_bpm_adm,114.0
caprefill_adm,No
agecalc_adm,49.4
spo2site2_pc_oxi_adm,99.0
glucose_mmolpl_adm,4.7
sqi2_perc_oxi_adm,97.0
sqi1_perc_oxi_adm,98.0


In [37]:
import pandas as pd

def calculate_bmi_zscore(sex_adm, agecalc_adm, height_cm_adm, weight_kg_adm, csv_path):
    """Calculate BMI-for-age z-score for children > 24 months."""
    bmi = weight_kg_adm / ((height_cm_adm / 100) ** 2)

    df = pd.read_csv(csv_path)
    df = df[df["Sex"] == (1 if sex_adm.lower() == "male" else 2)] 
    df = df.sort_values("Agemos")
    row = pd.merge_asof(pd.DataFrame({"Agemos": [agecalc_adm]}), df, on="Agemos", direction="nearest")

    L, M, S = row[['L', 'M', 'S']].values[0]
    return ((bmi / M) ** L - 1) / (L * S)


def calculate_wfl_zscore(height_cm_adm, weight_kg_adm, csv_path):
    """Calculate weight-for-length z-score for children ≤ 24 months."""
    df = pd.read_csv(csv_path)
    row = pd.merge_asof(pd.DataFrame({"Length(cm)": [height_cm_adm]}), df, on="Length(cm)", direction="nearest")

    L, M, S = row[['L', 'M', 'S']].values[0]
    return ((weight_kg_adm / M) ** L - 1) / (L * S)


def calculate_z_score(sex_adm, agecalc_adm, height_cm_adm, weight_kg_adm, csv_path=None):
    """
    Determines whether to calculate weight-for-length or BMI-for-age z-score 
    based on the child's age.

    Parameters:
    - sex_adm: Sex of the child ("Male" or "Female")
    - agecalc_adm: Age in months
    - height_cm_adm: Height in cm
    - weight_kg_adm: Weight in kg
    - csv_path: Optional path override for growth charts

    Returns:
    - z-score for the appropriate metric
    """
    if agecalc_adm > 24:
        csv_path = "charts/bmiagerev.csv"
        return calculate_bmi_zscore(sex_adm, agecalc_adm, height_cm_adm, weight_kg_adm, csv_path)
    else:
        csv_path = (
            "charts/WHO-Boys-Weight-for-length-Percentiles.csv" if sex_adm.lower() == "male" else \
            "charts/WHO-Girls-Weight-for-length-Percentiles.csv"
        )
        return calculate_wfl_zscore(height_cm_adm, weight_kg_adm, csv_path)


In [38]:
row

,sqi2_perc_oxi_adm,temp_c_adm,sqi1_perc_oxi_adm,hematocrit_gpdl_adm,spo2site1_pc_oxi_adm,diasbp_mmhg_adm,lactate_mmolpl_adm,caprefill_adm,hr_bpm_adm,sysbp_mmhg_adm,spo2site2_pc_oxi_adm,lactate2_mmolpl_adm,glucose_mmolpl_adm,inhospital_mortality
688,99.0,39.2,99.0,30.0,98.0,48.0,2.2,No,138.0,86.0,98.0,5.7,3.6,0


In [39]:
type(row)

pandas.core.frame.DataFrame

In [40]:
row.columns

Index(['sqi2_perc_oxi_adm', 'temp_c_adm', 'sqi1_perc_oxi_adm',
       'hematocrit_gpdl_adm', 'spo2site1_pc_oxi_adm', 'diasbp_mmhg_adm',
       'lactate_mmolpl_adm', 'caprefill_adm', 'hr_bpm_adm', 'sysbp_mmhg_adm',
       'spo2site2_pc_oxi_adm', 'lactate2_mmolpl_adm', 'glucose_mmolpl_adm',
       'inhospital_mortality'],
      dtype='object')